# 03 — Full Distributed BFS experiment matrix

Run only after Notebook 02 has completed and been reviewed. This notebook executes the full matrix:

- SNAP: `soc-Epinions1`, `web-Google`, `roadNet-CA`
- Synthetic R-MAT: scales 16 through 24, edge factor 8
- MPI ranks: 1, 2, 4, 8, 16
- Vertex orderings: original and fixed-seed Feistel relabelling
- Communication modes: blocking and non-blocking
- BFS source repetitions: 16 per configuration

Expected total: **240 configurations and 3,840 BFS traversals**.

The notebook probes Open MPI at every requested process count before the full run, checks Colab's available memory (including cgroup limits) before scale 24, and preserves validated configuration checkpoints in Drive for resume after a runtime reset.

Colab executes these MPI ranks on **one VM**; this is not multi-node cluster scaling. If the VM exposes fewer CPU cores than the requested rank count, those configurations are oversubscribed and the environment record will make that explicit.

**Success ZIP rule:** the complete matrix, every configuration's rank/validation records, all plots, the Markdown and DOCX reports and the PowerPoint must validate before the ZIP is created/downloaded.

In [3]:
from pathlib import Path
import os, sys, shutil, json, traceback, hashlib
from datetime import datetime, timezone
from google.colab import drive

drive.mount('/content/drive')
WORKSPACE = Path('/content/drive/MyDrive/Distributed_BFS_MPI_Colab')
SOURCE_DIR = WORKSPACE / 'Distributed_BFS_MPI'
WORK_DIR = Path('/content/Distributed_BFS_MPI')
PILOT_DIR = WORKSPACE / 'runs' / 'pilot'
RUN_DIR = WORKSPACE / 'runs' / 'full_matrix'
ARCHIVE_NAME = '03_full_experiment_results.zip'

# Remove any prior archive before checking prerequisites; a failed rerun must not
# leave a stale ZIP that could be mistaken for the current execution.
if (WORKSPACE / 'downloads').exists():
    (WORKSPACE / 'downloads' / ARCHIVE_NAME).unlink(missing_ok=True)
(Path('/content') / ARCHIVE_NAME).unlink(missing_ok=True)
sys.path.insert(0, str(WORKSPACE))
import colab_helpers as cb
cb.clear_previous_archive(WORKSPACE, ARCHIVE_NAME)
if not (SOURCE_DIR / 'src' / 'distributed_bfs.cpp').is_file():
    raise FileNotFoundError(f'Missing source project: {SOURCE_DIR}')
RUN_DIR.mkdir(parents=True, exist_ok=True)
(RUN_DIR / 'SUCCESS.json').unlink(missing_ok=True)
(RUN_DIR / 'FAILURE.json').unlink(missing_ok=True)
(RUN_DIR / 'RUN_COMPLETE.json').unlink(missing_ok=True)

# The reviewed pilot and Notebook 01 are mandatory prerequisites.
cb.require_correctness_passed(WORKSPACE, SOURCE_DIR)
cb.require_complete_run(PILOT_DIR, SOURCE_DIR)
if WORK_DIR.exists():
    shutil.rmtree(WORK_DIR)
shutil.copytree(SOURCE_DIR, WORK_DIR)

FULL_RANKS = [1, 2, 4, 8, 16]
FULL_SCALES = list(range(16, 25))
FULL_REPETITIONS = 16
FULL_EDGE_FACTOR = 8
MIN_FREE_RAM_GB_FOR_SCALE24 = 9.0

free_ram_gb = cb.available_memory_gb()
logical_cpus = os.cpu_count() or 1
print(f'Available memory estimate (host + cgroup): {free_ram_gb:.2f} GiB')
print(f'Logical CPU count reported by Colab VM: {logical_cpus}')
print(f'MPI rank counts requested: {FULL_RANKS}')
if max(FULL_RANKS) > logical_cpus:
    print('WARNING: some MPI configurations will be oversubscribed; interpret scaling results accordingly.')
if free_ram_gb < MIN_FREE_RAM_GB_FOR_SCALE24:
    raise MemoryError(
        f'Scale 24 is enabled and only {free_ram_gb:.2f} GiB is available. '
        f'This notebook requires at least {MIN_FREE_RAM_GB_FOR_SCALE24:.1f} GiB free before starting. '
        'Switch to a High-RAM Colab runtime, reconnect, and rerun.'
    )
print('Drive result folder:', RUN_DIR)
print('Local source copy:', WORK_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Available memory estimate (host + cgroup): 11.47 GiB
Logical CPU count reported by Colab VM: 2
MPI rank counts requested: [1, 2, 4, 8, 16]
Drive result folder: /content/drive/MyDrive/Distributed_BFS_MPI_Colab/runs/full_matrix
Local source copy: /content/Distributed_BFS_MPI


## Prepare SNAP cache, validate Open MPI process counts, then run the full matrix

If a runtime disconnects, rerun this notebook with the same settings. The `run_config.json` freezes the run settings/source hash; valid `.done` checkpoints are reused, and incomplete or invalid configurations are rerun.

In [4]:
try:
    # Re-probe 1/2/4/8/16 ranks in this runtime. This is a tiny MPI program and
    # catches process-launch failures before the expensive BFS matrix starts.
    mpi = cb.install_openmpi_and_build(
        WORK_DIR, WORKSPACE, process_counts=FULL_RANKS,
        install=True, install_python_requirements=True,
    )
    metadata = RUN_DIR / 'metadata'
    metadata.mkdir(parents=True, exist_ok=True)
    (metadata / 'openmpi_toolchain.json').write_text(json.dumps(mpi['report'], indent=2), encoding='utf-8')

    # Ensure all scripts are executable to avoid Permission Denied errors
    for script in (WORK_DIR / 'scripts').glob('*.sh'):
        script.chmod(0o755)

    # Restore persistent SNAP input files from Drive; download only missing files.
    drive_snap = WORKSPACE / 'data' / 'snap'
    local_snap = WORK_DIR / 'data' / 'snap'
    drive_snap.mkdir(parents=True, exist_ok=True)
    local_snap.mkdir(parents=True, exist_ok=True)
    expected_snap = ['soc-Epinions1.txt', 'web-Google.txt', 'roadNet-CA.txt']
    for name in expected_snap:
        cached = drive_snap / name
        if cached.is_file() and cached.stat().st_size > 0:
            shutil.copy2(cached, local_snap / name)
    missing = [name for name in expected_snap if not (local_snap / name).is_file() or (local_snap / name).stat().st_size == 0]
    if missing:
        print('SNAP files missing from Drive cache; downloading:', missing)
        cb.run_stream(
            ['bash', str(WORK_DIR / 'scripts' / 'download_snap_graphs.sh')],
            cwd=WORK_DIR, env=mpi['env'], timeout=1800, check=True,
        )
    for name in expected_snap:
        path = local_snap / name
        if not path.is_file() or path.stat().st_size == 0:
            raise FileNotFoundError(f'SNAP input missing or empty after download: {path}')
        shutil.copy2(path, drive_snap / name)
    cache_manifest = {
        name: {'bytes': (drive_snap / name).stat().st_size,
               'sha256': hashlib.sha256((drive_snap / name).read_bytes()).hexdigest()}
        for name in expected_snap
    }
    (RUN_DIR / 'snap_cache_manifest.json').write_text(json.dumps(cache_manifest, indent=2), encoding='utf-8')

    run_settings = {
        'stage': 'full_matrix',
        'datasets': ['soc-Epinions1', 'web-Google', 'roadNet-CA'] + [f'RMAT_s{x}' for x in FULL_SCALES],
        'ranks': FULL_RANKS,
        'orderings': ['original', 'random'],
        'modes': ['blocking', 'nonblocking'],
        'repetitions': FULL_REPETITIONS,
        'scales': FULL_SCALES,
        'edge_factor': FULL_EDGE_FACTOR,
        'run_snap': True,
        'run_rmat': True,
        'mpi_compiler': mpi['mpicxx'],
        'mpi_launcher': mpi['mpirun'],
        'mpi_version': mpi['report']['launcher_version'].splitlines()[0],
        'snap_sha256': {k: v['sha256'] for k, v in cache_manifest.items()},
    }
    cb.ensure_run_config(RUN_DIR, run_settings, SOURCE_DIR)

    env = mpi['env'].copy()
    env.update({
        'OUT_DIR': str(RUN_DIR),
        'RANKS': '1 2 4 8 16',
        'REPETITIONS': str(FULL_REPETITIONS),
        'SCALES': '16 17 18 19 20 21 22 23 24',
        'EDGE_FACTOR': str(FULL_EDGE_FACTOR),
        'RUN_SNAP': '1',
        'RUN_RMAT': '1',
        'RESUME': '1',
        'MPIEXEC': mpi['mpirun'],
        'MPICXX': mpi['mpicxx'],
        'MPIEXEC_FLAGS': '--allow-run-as-root --oversubscribe --bind-to none',
        'RUNNER_VERSION': cb.RUNNER_VERSION,
        'SOURCE_FINGERPRINT': cb.source_fingerprint(SOURCE_DIR),
    })
    cb.run_stream(
        ['bash', str(WORK_DIR / 'scripts' / 'run_experiments.sh')],
        cwd=WORK_DIR, env=env, log_path=RUN_DIR / 'colab_runner.log',
        timeout=None, check=True,
    )

    expected_datasets = ['soc-Epinions1', 'web-Google', 'roadNet-CA'] + [f'RMAT_s{x}' for x in FULL_SCALES]
    completion = cb.validate_complete_run(
        RUN_DIR, datasets=expected_datasets, ranks=FULL_RANKS,
        orderings=['original', 'random'], modes=['blocking', 'nonblocking'],
        repetitions=FULL_REPETITIONS,
    )
    if completion['configurations'] != 240 or completion['bfs_summary_rows'] != 3840:
        raise RuntimeError(f'Full-matrix totals do not match expected 240 configurations / 3840 BFS runs: {completion}')

    cb.write_success_manifest(RUN_DIR, '03', {
        'source_fingerprint': cb.source_fingerprint(SOURCE_DIR),
        'completion': completion,
        'mpi_implementation': 'Open MPI',
        'available_memory_gib_before_run': free_ram_gb,
        'logical_cpus': logical_cpus,
    })
    cb.create_success_zip(RUN_DIR, WORKSPACE, ARCHIVE_NAME)
    print('\nFULL MATRIX COMPLETE: 240 configurations / 3,840 BFS traversals.')
    print('Automatically downloaded:', ARCHIVE_NAME)

except BaseException as exc:
    (RUN_DIR / 'SUCCESS.json').unlink(missing_ok=True)
    (RUN_DIR / 'FAILURE.json').write_text(json.dumps({
        'success': False, 'notebook': '03',
        'error_type': type(exc).__name__, 'error': str(exc),
        'traceback': traceback.format_exc(),
        'failed_at_utc': datetime.now(timezone.utc).isoformat(),
    }, indent=2), encoding='utf-8')
    print('\nFAILED/INCOMPLETE: no full-matrix ZIP was created.')
    print('Drive retains valid checkpoints and diagnostics in:', RUN_DIR)
    print('Rerun this notebook with unchanged settings to resume.')
    raise


$ /usr/bin/mpicxx.openmpi -O2 /content/mpi_two_rank_probe.cpp -o /content/mpi_two_rank_probe
Verified Open MPI -np 1: ranks [0], communicator size 1.
Verified Open MPI -np 2: ranks [0, 1], communicator size 2.
Verified Open MPI -np 4: ranks [0, 1, 2, 3], communicator size 4.
Verified Open MPI -np 8: ranks [0, 1, 2, 3, 4, 5, 6, 7], communicator size 8.
Verified Open MPI -np 16: ranks [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15], communicator size 16.

$ python3 -m pip install -q -r /content/Distributed_BFS_MPI/requirements.txt

$ bash /content/Distributed_BFS_MPI/scripts/build.sh
In file included from /usr/lib/x86_64-linux-gnu/openmpi/include/openmpi/ompi/mpi/cxx/mpicxx.h:277,
                 from /usr/lib/x86_64-linux-gnu/openmpi/include/mpi.h:2911,
                 from /content/Distributed_BFS_MPI/src/distributed_bfs.cpp:1:
/usr/lib/x86_64-linux-gnu/openmpi/include/openmpi/ompi/mpi/cxx/op_inln.h: In member function ‘virtual void MPI::Op::Init(void (*)(const void*, void*, 

RuntimeError: Command failed with exit code 137: ['bash', '/content/Distributed_BFS_MPI/scripts/run_experiments.sh']
[SKIP validated checkpoint] RMAT_s20_p1_random_blocking
[SKIP validated checkpoint] RMAT_s20_p1_random_nonblocking
[SKIP validated checkpoint] RMAT_s20_p2_original_blocking
[SKIP validated checkpoint] RMAT_s20_p2_original_nonblocking
[SKIP validated checkpoint] RMAT_s20_p2_random_blocking
[SKIP validated checkpoint] RMAT_s20_p2_random_nonblocking
[SKIP validated checkpoint] RMAT_s20_p4_original_blocking
[SKIP validated checkpoint] RMAT_s20_p4_original_nonblocking
[SKIP validated checkpoint] RMAT_s20_p4_random_blocking
[SKIP validated checkpoint] RMAT_s20_p4_random_nonblocking
[SKIP validated checkpoint] RMAT_s20_p8_original_blocking
[SKIP validated checkpoint] RMAT_s20_p8_original_nonblocking
[SKIP validated checkpoint] RMAT_s20_p8_random_blocking
[SKIP validated checkpoint] RMAT_s20_p8_random_nonblocking
[SKIP validated checkpoint] RMAT_s20_p16_original_blocking
[SKIP v

## Full-matrix output

On success, Colab automatically downloads `03_full_experiment_results.zip` and saves another copy under `MyDrive/Distributed_BFS_MPI_Colab/downloads/`.

If Colab disconnects or any configuration/report validation fails, no success ZIP is created. Rerun this notebook with the same configuration to continue from valid checkpoints. Do not change the ranks, scales, repetitions, edge factor or SNAP files during a resumed run.